In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv(
    "../data/processed/ev_sales_cleaned.csv",
    parse_dates=["Date"]
)

df.head()

,Date,Year,Month,State,Vehicle_Category,EV_Sales
0,2021-01-01,2021,1,Delhi,Electric 2W,2300
1,2021-01-01,2021,1,Delhi,Electric 3W,2100
2,2021-01-01,2021,1,Delhi,Electric 4W,700
3,2021-01-01,2021,1,Karnataka,Electric 2W,3100
4,2021-01-01,2021,1,Karnataka,Electric 3W,1200


## Sort the Dataset
- This is essential before creating lag features.

In [3]:
df = df.sort_values(
    ["State", "Vehicle_Category", "Date"]
).reset_index(drop=True)

**Step 4: Create Time-Based Features**

In [4]:
df["Quarter"] = df["Date"].dt.quarter

df["Month_Name"] = df["Date"].dt.month_name()

df["Year_Month"] = df["Date"].dt.to_period("M").astype(str)

In [5]:
df

,Date,Year,Month,State,Vehicle_Category,EV_Sales,Quarter,Month_Name,Year_Month
0,2021-01-01,2021,1,Delhi,Electric 2W,2300,1,January,2021-01
1,2021-02-01,2021,2,Delhi,Electric 2W,2450,1,February,2021-02
2,2021-03-01,2021,3,Delhi,Electric 2W,2860,1,March,2021-03
3,2021-04-01,2021,4,Delhi,Electric 2W,2700,2,April,2021-04
4,2021-05-01,2021,5,Delhi,Electric 2W,3020,2,May,2021-05
...,...,...,...,...,...,...,...,...,...
283,2022-08-01,2022,8,Uttar Pradesh,Electric 4W,1840,3,August,2022-08
284,2022-09-01,2022,9,Uttar Pradesh,Electric 4W,2020,3,September,2022-09
285,2022-10-01,2022,10,Uttar Pradesh,Electric 4W,2300,4,October,2022-10
286,2022-11-01,2022,11,Uttar Pradesh,Electric 4W,2520,4,November,2022-11


**Step 5: Create Lag Features**
- These allow the model to use previous months' sales.

In [6]:
df["Lag_1"] = (
    df.groupby(["State", "Vehicle_Category"])["EV_Sales"]
      .shift(1)
)

df["Lag_2"] = (
    df.groupby(["State", "Vehicle_Category"])["EV_Sales"]
      .shift(2)
)

df["Lag_3"] = (
    df.groupby(["State", "Vehicle_Category"])["EV_Sales"]
      .shift(3)
)

**Step 6: Create Rolling Mean**

In [7]:
df["Rolling_3"] = (
    df.groupby(["State", "Vehicle_Category"])["EV_Sales"]
      .transform(lambda x: x.rolling(3).mean())
)

In [8]:
df["Rolling_STD_3"] = (
    df.groupby(["State", "Vehicle_Category"])["EV_Sales"]
      .transform(lambda x: x.rolling(3).std())
)

In [9]:
df["MoM_Growth"] = (
    df.groupby(["State", "Vehicle_Category"])["EV_Sales"]
      .pct_change() * 100
)

**Step 9: Handle Missing Values**
The first few rows in each state/vehicle group will have missing lag and rolling values. Fill them with 0 for the ML models.

In [10]:
df = df.fillna(0)

In [11]:
df.isnull().sum()

Date                0
Year                0
Month               0
State               0
Vehicle_Category    0
EV_Sales            0
Quarter             0
Month_Name          0
Year_Month          0
Lag_1               0
Lag_2               0
Lag_3               0
Rolling_3           0
Rolling_STD_3       0
MoM_Growth          0
dtype: int64

**Step 10: Verify the New Features**

In [12]:
df.head(10)

,Date,Year,Month,State,Vehicle_Category,EV_Sales,Quarter,Month_Name,Year_Month,Lag_1,Lag_2,Lag_3,Rolling_3,Rolling_STD_3,MoM_Growth
0,2021-01-01,2021,1,Delhi,Electric 2W,2300,1,January,2021-01,0.0,0.0,0.0,0.000000,0.000000,0.000000
1,2021-02-01,2021,2,Delhi,Electric 2W,2450,1,February,2021-02,2300.0,0.0,0.0,0.000000,0.000000,6.521739
2,2021-03-01,2021,3,Delhi,Electric 2W,2860,1,March,2021-03,2450.0,2300.0,0.0,2536.666667,289.885035,16.734694
3,2021-04-01,2021,4,Delhi,Electric 2W,2700,2,April,2021-04,2860.0,2450.0,2300.0,2670.000000,206.639783,-5.594406
4,2021-05-01,2021,5,Delhi,Electric 2W,3020,2,May,2021-05,2700.0,2860.0,2450.0,2860.000000,160.000000,11.851852
5,2021-06-01,2021,6,Delhi,Electric 2W,3250,2,June,2021-06,3020.0,2700.0,2860.0,2990.000000,276.224546,7.615894
6,2021-07-01,2021,7,Delhi,Electric 2W,3400,3,July,2021-07,3250.0,3020.0,2700.0,3223.333333,191.398363,4.615385
7,2021-08-01,2021,8,Delhi,Electric 2W,3620,3,August,2021-08,3400.0,3250.0,3020.0,3423.333333,186.100331,6.470588
8,2021-09-01,2021,9,Delhi,Electric 2W,3890,3,September,2021-09,3620.0,3400.0,3250.0,3636.666667,245.424802,7.458564
9,2021-10-01,2021,10,Delhi,Electric 2W,4350,4,October,2021-10,3890.0,3620.0,3400.0,3953.333333,369.098000,11.825193


In [13]:
df.to_csv(
    "../data/processed/ev_sales_feature_engineered.csv",
    index=False
)